In [1]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType
)
from pyspark.sql.functions import col, from_json, to_json, coalesce
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, from_json


In [2]:
spark = (
    SparkSession.builder
    .appName("silver-cdc")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo.type", "rest")
    .config("spark.sql.catalog.demo.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.demo.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.demo.s3.access-key-id", "admin")
    .config("spark.sql.catalog.demo.s3.secret-access-key", "admin123")
    .config("spark.sql.catalog.demo.s3.path-style-access", "true")
    .config("spark.sql.catalog.demo.s3.region", "us-east-1")
    .getOrCreate()
)

In [3]:
spark.sql("SELECT COUNT(*) FROM demo.bronze.customers_cdc").show()

+--------+
|count(1)|
+--------+
|   21911|
+--------+



In [4]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.silver")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.customers (
    id BIGINT,
    name STRING,
    email STRING,
    created_at BIGINT
)
USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.silver.drivers (
    id BIGINT,
    name STRING,
    license_number STRING,
    created_at BIGINT
)
USING iceberg
""")

DataFrame[]

In [5]:
customers_bronze = spark.table("demo.bronze.customers_cdc")
drivers_bronze   = spark.table("demo.bronze.drivers_cdc")

In [6]:
window = Window.partitionBy("pk_id").orderBy(col("event_ts_ms").desc())

customers_latest = (
    customers_bronze
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

drivers_latest = (
    drivers_bronze
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [7]:
customer_schema = StructType([
    StructField("id", LongType()),
    StructField("name", StringType()),
    StructField("email", StringType()),
    StructField("created_at", LongType()),
])

customers_upserts = (
    customers_latest
    .filter(col("op").isin("c", "u", "r"))
    .withColumn("after", from_json(col("after_json"), customer_schema))
    .select(
        col("after.id").alias("id"),
        col("after.name").alias("name"),
        col("after.email").alias("email"),
        col("after.created_at").alias("created_at")
    )
)

customers_deletes = (
    customers_latest
    .filter(col("op") == "d")
    .select(col("pk_id").alias("id"))
)

In [9]:
spark.sql("DROP TABLE IF EXISTS demo.silver.customers_upserts_stage")

customers_upserts.writeTo("demo.silver.customers_upserts_stage").create()

In [10]:
spark.sql("""
MERGE INTO demo.silver.customers AS t
USING demo.silver.customers_upserts_stage AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET
    t.name = s.name,
    t.email = s.email,
    t.created_at = s.created_at
WHEN NOT MATCHED THEN INSERT (
    id, name, email, created_at
) VALUES (
    s.id, s.name, s.email, s.created_at
)
""")

DataFrame[]

In [11]:
spark.sql("DROP TABLE IF EXISTS demo.silver.customers_deletes_stage")

customers_deletes.writeTo("demo.silver.customers_deletes_stage").create()

In [12]:
spark.sql("SELECT COUNT(*) FROM demo.silver.customers").show()
spark.sql("SELECT * FROM demo.silver.customers LIMIT 10").show()

+--------+
|count(1)|
+--------+
|    3809|
+--------+

+----+--------------+--------------------+----------------+
|  id|          name|               email|      created_at|
+----+--------------+--------------------+----------------+
|1697|Anna Johansson|updated_1697_933@...|1776622211339947|
|1950| Ivan Virtanen|updated_1950_911@...|1776659806430979|
|2250|  Lucas Larsen|updated_2250_836@...|1776683839578739|
|2927|Mateo Nakamura|updated_2927_736@...|1776741490340753|
|3091| Raj Johansson|raj.johansson@exa...|1776786286416912|
|3764|   Mateo Smith|updated_3764_228@...|1776884447383322|
|4590|   Amir Petrov|amir.petrov@examp...|1776948859015996|
|4823|Noah Johansson|updated_4823_353@...|1776959332281156|
|4894|    Luna Kumar|updated_4894_857@...|1776962411664756|
|5409|    Lars Kumar|lars.kumar@exampl...|1776967883861608|
+----+--------------+--------------------+----------------+



In [13]:
drivers_bronze = spark.table("demo.bronze.drivers_cdc")

window = Window.partitionBy("pk_id").orderBy(col("event_ts_ms").desc())

drivers_latest = (
    drivers_bronze
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [14]:
driver_schema = StructType([
    StructField("id", LongType()),
    StructField("name", StringType()),
    StructField("license_number", StringType()),
    StructField("created_at", LongType()),
])

drivers_upserts = (
    drivers_latest
    .filter(col("op").isin("c", "u", "r"))
    .withColumn("after", from_json(col("after_json"), driver_schema))
    .select(
        col("after.id").alias("id"),
        col("after.name").alias("name"),
        col("after.license_number").alias("license_number"),
        col("after.created_at").alias("created_at")
    )
)

drivers_deletes = (
    drivers_latest
    .filter(col("op") == "d")
    .select(col("pk_id").alias("id"))
)

In [15]:
spark.sql("DROP TABLE IF EXISTS demo.silver.drivers_upserts_stage")
drivers_upserts.writeTo("demo.silver.drivers_upserts_stage").create()

In [16]:
spark.sql("""
MERGE INTO demo.silver.drivers AS t
USING demo.silver.drivers_upserts_stage AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET
    t.name = s.name,
    t.license_number = s.license_number,
    t.created_at = s.created_at
WHEN NOT MATCHED THEN INSERT (
    id, name, license_number, created_at
) VALUES (
    s.id, s.name, s.license_number, s.created_at
)
""")

DataFrame[]

In [17]:
spark.sql("DROP TABLE IF EXISTS demo.silver.drivers_deletes_stage")
drivers_deletes.writeTo("demo.silver.drivers_deletes_stage").create()

In [18]:
spark.sql("""
DELETE FROM demo.silver.drivers
WHERE id IN (
    SELECT id FROM demo.silver.drivers_deletes_stage
)
""")

DataFrame[]

In [19]:
spark.sql("SELECT COUNT(*) FROM demo.silver.drivers").show()
spark.sql("SELECT * FROM demo.silver.drivers LIMIT 10").show()

+--------+
|count(1)|
+--------+
|    1387|
+--------+

+----+---------------+--------------+----------------+
|  id|           name|license_number|      created_at|
+----+---------------+--------------+----------------+
|1950|    Anna Petrov|     TLC-85479|1776885349907764|
|2250|    Lars Garcia|     TLC-29629|1776947706128870|
|2927|    Maria Silva|     TLC-54409|1776969858739756|
|3091| Amir Johansson|     TLC-63116|1776971262156252|
|3764|Olivia Virtanen|     TLC-99509|1777192054891396|
|2895|     Kai Larsen|     TLC-25026|1776969574848050|
|2941|  Zara Kowalski|     TLC-50460|1776969946416849|
|3009|    Lars Tanaka|     TLC-45656|1776970573349342|
|3015|    Amir Larsen|     TLC-86632|1776970649994809|
|3106|     Luna Novak|     TLC-45230|1776976160092740|
+----+---------------+--------------+----------------+



In [21]:
spark.sql("SELECT COUNT(*) FROM demo.silver.customers").show()
spark.sql("SELECT COUNT(*) FROM demo.silver.drivers").show()

+--------+
|count(1)|
+--------+
|    3809|
+--------+

+--------+
|count(1)|
+--------+
|    1387|
+--------+

